### **Parte 5: Proyección I**

caegamos dataset y herramientas para crear el flujo automatizado

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de Machine Learning (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Cargar el dataset final
df = pd.read_csv('../data/processed/movies_listo_para_modelo.csv')

# Limpieza de seguridad: eliminar valores infinitos si hubo divisiones por cero en el ROI
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['roi', 'budget', 'temporada'])

print(f"Datos cargados listos para entrenar: {df.shape[0]} películas.")

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/movies_listo_para_modelo.csv'

separamos las columnas que van a "adivinar" (X) de la columna que queremos predecir (y). Luego, guardamos un 20% de los datos como "Prueba" para evaluar si el modelo realmente aprendió o si solo memorizó.

In [ ]:
# 1. Definir la variable objetivo (Lo que queremos predecir)
y = df['roi']

# 2. Definir las variables predictoras (Nuestras características de la Fase 4)
features = ['budget', 'popularity', 'temporada', 'es_drama', 'es_comedia', 'es_thriller']
X = df[features]

# 3. Dividir en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape[0]} | Datos de prueba: {X_test.shape[0]}")

El ColumnTransformer estandariza los números grandes (como el presupuesto) y codifica la columna de texto temporada en ceros y unos . Luego, el Pipeline conecta esa transformación directamente con el algoritmo de Regresión Lineal.

In [ ]:
# 1. Separar columnas por tipo para tratarlas distinto
num_features = ['budget', 'popularity', 'es_drama', 'es_comedia', 'es_thriller']
cat_features = ['temporada']

# 2. Crear el transformador de columnas (Escalar números, Codificar texto)
preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

# 3. Ensamblar el Pipeline final
modelo_pipeline = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('algoritmo', LinearRegression())
])

# 4. ENTRENAR EL MODELO
modelo_pipeline.fit(X_train, y_train)

print("Pipeline ejecutado y Modelo entrenado exitosamente.")

Ponemos a prueba a la IA con los datos que no conoce (X_test). Usamos la métrica $R^2$ (R-cuadrado) para ver qué porcentaje de las ganancias se explica por las variables que le dimos.

In [ ]:
# 1. Hacer predicciones con los datos de prueba
y_pred = modelo_pipeline.predict(X_test)

# 2. Calcular métricas de error
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("---  RESULTADOS DE LA PROYECCIÓN ---")
print(f"Error Cuadrático Medio (MSE): {mse:.2f}")
print(f"Precisión del Modelo (R2 Score): {r2:.4f}")

Este gráfico muestra qué variables tuvieron un impacto positivo (barras hacia la derecha) o negativo (barras hacia la izquierda) en la rentabilidad de las películas.

In [ ]:
# Extraer los nombres de las columnas después del OneHotEncoder
cat_encoder = modelo_pipeline.named_steps['preprocesador'].transformers_[1][1]
cat_nombres = cat_encoder.get_feature_names_out(cat_features)
nombres_finales = num_features + list(cat_nombres)

# Extraer el "peso" (coeficientes) que el modelo le dio a cada variable
coeficientes = modelo_pipeline.named_steps['algoritmo'].coef_

# Crear un DataFrame para graficar
df_importancia = pd.DataFrame({
    'Variable': nombres_finales,
    'Impacto en ROI': coeficientes
}).sort_values(by='Impacto en ROI', ascending=False)

# Graficar
plt.figure(figsize=(10, 6))
sns.barplot(x='Impacto en ROI', y='Variable', data=df_importancia, palette='viridis')
plt.title('Impacto del Timing y Género en la Rentabilidad (ROI)', fontsize=14)
plt.axvline(0, color='black', linestyle='--')
plt.xlabel('Impacto Promedio en el ROI')
plt.ylabel('Característica')
plt.tight_layout()
plt.savefig('../outputs/figures/impacto_variables.png')
plt.show()